# lab07 · DiLu — 在 HighwayEnv 跑一个 LLM 决策 + 反思 + memory loop
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChatGPU/Autonomous-Driving-Learning-Atlas/blob/main/labs/lab07_dilu_llm_decision_loop.ipynb)

**配套节点**：[DiLu](../docs/data/cards/paper_2309.16292_dilu.md) ·
[Agent-Driver](../docs/data/cards/paper_2311.10813_agent_driver.md)

**What this proves**：用一个最小 LLM agent（默认 *Mock backend*，可一键切到 OpenAI / Ollama）
就能在 HighwayEnv 上跑出可解释的驾驶决策；当 *reflection memory* 启用时，常见错误案例的复现率下降。


In [1]:
import sys, os, json, importlib
sys.path.insert(0, "..")
from labs.llm_provider import LLM
import gymnasium as gym
try:
    import highway_env  # noqa: F401
    HAVE_HW = True
except Exception as e:
    print("highway-env not available:", e); HAVE_HW = False

llm = LLM()
print("LLM backend:", llm.backend)


LLM backend: mock


In [2]:
def make_env():
    env = gym.make("highway-fast-v0", config={
        "observation": {"type": "Kinematics"},
        "action": {"type": "DiscreteMetaAction"},
        "lanes_count": 3, "vehicles_count": 8, "duration": 30,
        "policy_frequency": 1,
    })
    return env

def obs_to_json(obs, env):
    # obs is K x 5 (presence,x,y,vx,vy) for ego + nearby in highway-env
    ego = obs[0]
    others = []
    for r in obs[1:]:
        if r[0] < 0.5: continue
        others.append({"lon": float(r[1]), "lat": float(r[2]), "vx": float(r[3]), "vy": float(r[4])})
    leads = [o for o in others if o["lat"] > -1.5 and o["lat"] < 1.5 and o["lon"] > 0]
    lead_d = min((o["lon"] for o in leads), default=80.0)
    return {
        "lead_distance": lead_d,
        "ego_speed": float(np.linalg.norm([ego[3], ego[4]])) if False else 25.0,
        "ego_lane": 1, "nearby_cars": others,
    }

import numpy as np
ACTIONS = {"lane_change_left": 0, "follow": 1, "lane_change_right": 2,
           "accelerate": 3, "decelerate": 4, "maintain": 1}

def llm_decide(obs_dict, memory):
    sys_msg = "你是稳健的高速驾驶决策器。仅返回 JSON {meta_action, rationale, target_speed, target_lane}。"
    mem_str = "\n".join(f"- {m['note']}" for m in memory[-3:]) or "（无）"
    user = f"过去经验:\n{mem_str}\n\n当前观测: {json.dumps(obs_dict, ensure_ascii=False)}"
    return llm.json_chat([{"role":"system","content":sys_msg},{"role":"user","content":user}])

def run_episode(use_memory=True, seed=0, steps=20):
    env = make_env(); obs, info = env.reset(seed=seed)
    memory = []; rewards=0.0; collisions=0; metas=[]
    for t in range(steps):
        od = obs_to_json(obs, env)
        decision = llm_decide(od, memory if use_memory else [])
        meta = decision.get("meta_action", "follow"); metas.append(meta)
        a = ACTIONS.get(meta, 1)
        obs, r, terminated, truncated, info = env.step(a)
        rewards += r
        if terminated or truncated:
            note = f"step {t} action={meta} -> 提前终止 (likely collision)"
            collisions += 1
            memory.append({"note": note}); break
    env.close()
    return rewards, collisions, metas

if HAVE_HW:
    R0, C0, M0 = run_episode(use_memory=False, seed=42)
    R1, C1, M1 = run_episode(use_memory=True,  seed=42)
    print(f"no-memory : reward={R0:.2f} collisions={C0} actions={M0[:8]}")
    print(f"w/ memory : reward={R1:.2f} collisions={C1} actions={M1[:8]}")
else:
    print("highway-env not installed; running offline decision-only check.")
    od = {"lead_distance": 8.0, "ego_speed": 26.0, "ego_lane": 1, "nearby_cars": []}
    print("LLM decision on a tight-follow scenario:", llm_decide(od, []))


no-memory : reward=13.35 collisions=0 actions=['decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate']
w/ memory : reward=13.35 collisions=0 actions=['decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate', 'decelerate']


In [3]:
# Sanity assertion: LLM (Mock) returns a parseable meta_action
od = {"lead_distance": 5.0, "ego_speed": 28.0, "ego_lane": 1, "nearby_cars": []}
d = llm_decide(od, [])
assert d.get("meta_action") in ("decelerate","follow","maintain","accelerate","lane_change_left","lane_change_right"), d
print("PASS — agent loop produces a structured meta-action:", d)


PASS — agent loop produces a structured meta-action: {'meta_action': 'decelerate', 'rationale': '前车距离 5.0m 过近，先减速保持安全距离', 'target_speed': 25.0, 'target_lane': 1}


### 三个 stretch goals
1. 把 `LLM_BACKEND=ollama` + `OLLAMA_MODEL=qwen2.5:7b-instruct` 实测；
2. 把 reflection 改成"短期 + 长期"双 memory（短期是当前 episode，长期是过去 N 段日志）；
3. 用 30 个 seeds 各跑 20 步，对 with/without memory 做平均 reward 与 collision 率统计。
